idk the other one was getting weird

In [1]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
import array

In [2]:
ROOT.gDirectory.Clear()

yield_file_0mb = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_nch60/merged_yields.root"
yield_file_3mb = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/merged_yields.root"

In [29]:
high_bins = [ [60,71], [71,78], [78,91], [91,97], [97,1000] ]

analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [30]:
eta_min = 2 #from prl paper, abs(delta eta star) should be greater than 2
eta_max = 4 #all the delta eta values should be less than this anyways i think

def project(hYield, suffix):
    '''
    Makes the 1d projection of the yield histograms
    input: yield histogram
    output: 1d projection
    '''
    # 1. Bins within delta eta star range
    bin_low = hYield.GetXaxis().FindBin(eta_min)
    bin_high = hYield.GetXaxis().FindBin(eta_max)
    
    # 2. doing the projection onto delta phi star
    h1D = hYield.Clone()
    h1D = h1D.ProjectionY(f"h1D_{suffix}", bin_low, bin_high)

    # 3. normalising by delta phi star bin width
    phi_bw = h1D.GetBinWidth(1)
    h1D.Scale(1.0 / phi_bw)

    # -------------------------------------------------------------------------
    # CRITICAL FIX: Convert raw X-axis bin sum to a Delta-Eta average
    # -------------------------------------------------------------------------
    #eta_bw = hYield.GetXaxis().GetBinWidth(1) # e.g., 0.1
    #eta_width = eta_max - eta_min             # 4 - 2 = 2.0
    
    #h1D.Scale(eta_bw / eta_width) 
    # -------------------------------------------------------------------------

    return h1D



#Fourier decomposition of the projection

#fourier fitting

#cosine series (param [0] should be N_assoc)
cosine_series = (
    "[0]/(2*TMath::Pi()) * (1 + 2*[1]*TMath::Cos(x) + 2*[2]*TMath::Cos(2*x) "
    "+ 2*[3]*TMath::Cos(3*x) + 2*[4]*TMath::Cos(4*x) + 2*[5]*TMath::Cos(5*x))"
)



def fourierFit(h1D, prefix, Nassoc = None):
    '''
    Applies a fourier fit to the 1d projection
    
    inputs: the 1d projection, and optional Nassoc
    output: the fourier coefficients and their errors. 

    modifications: the 1d histogram gets the fourier fit function put into it
    '''

    # root fit function
    fit_func = ROOT.TF1(f"fourier_fit_{prefix}", cosine_series, -0.5*math.pi , 1.5*math.pi)

    #seeding parameters
    if Nassoc is not None:
        fit_func.SetParameter(0, Nassoc)    #what it should be, based on the paper
    else:
        # "width" option calculates the true area under your 1D curve (height * 2pi)
        # This provides a mathematically perfect initial seed for parameter [0]
        fit_func.SetParameter(0, h1D.Integral("width"))
        
    fit_func.SetParameter(1, 0.1)   # v1 seed
    fit_func.SetParameter(2, 0.1)   # v2 seed (elliptic flow)
    fit_func.SetParameter(3, 0.1)   # v3 seed
    fit_func.SetParameter(4, 0.1)   # v4 seed
    fit_func.SetParameter(5, 0.1)   # v5 seed       all the seeds are 0.1 in the DrawFlow.C file so I'm just using that


    # EXECUTE THE FIT
    # ==========================================
    # "R" forces the fit range specified in the TF1 definition
    # "M" tells ROOT to search for better minimums (improves v2 precision)
    # "E" invokes the advanced Minos error estimation
    # "Q" keeps the terminal output quiet
    h1D.Fit(fit_func, "R M E Q")

    #extract parameters
    v1 = fit_func.GetParameter(1)
    v2 = fit_func.GetParameter(2)  # This is the fourier coefficient
    v3 = fit_func.GetParameter(3)

    #get error
    v1_err = fit_func.GetParError(1)
    v2_err = fit_func.GetParError(2)
    v3_err = fit_func.GetParError(3)

    #this bit is in the github but idk if i need it:
    # Apply the sqrt(2) scaling factor found in DrawVn's source code to conservatively account for Signal/Background statistical correlation
    v2_err_final = v2_err * math.sqrt(2)

    coeffs = [v1, v2, v3]
    errs = [v1_err, v2_err, v3_err]
    return v2, v2_err


In [31]:

def openYields(filepath):

    wta_yields = {}
    std_yields = {}

    wta_avg_Nch = {}
    std_avg_Nch = {}

    file = ROOT.TFile.Open(filepath, "READ")

    for mult_bin in high_bins:
        bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
        
        hYield_wta = file.Get(f"WTA_yield_{mult_bin[0]}_{mult_bin[1]}")
        hYield_std = file.Get(f"STD_yield_{mult_bin[0]}_{mult_bin[1]}")

        hYield_wta.SetDirectory(0)
        hYield_std.SetDirectory(0)

        wta_yields[bin_key] = hYield_wta
        std_yields[bin_key] = hYield_std

        bin_wta_avg_Nch = file.Get(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        bin_std_avg_Nch = file.Get(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        wta_avg_Nch[bin_key] = np.float64(bin_wta_avg_Nch)
        std_avg_Nch[bin_key] = np.float64(bin_std_avg_Nch)

    return wta_yields, std_yields, wta_avg_Nch, std_avg_Nch

In [32]:
wta_yields_0mb, std_yields_0mb, wta_avg_Nch_0mb, std_avg_Nch_0mb = openYields(yield_file_0mb)
wta_yields_3mb, std_yields_3mb, wta_avg_Nch_3mb, std_avg_Nch_3mb = openYields(yield_file_3mb)

In [33]:
print(wta_avg_Nch_0mb)
print(std_avg_Nch_0mb)
print(wta_avg_Nch_3mb)
print(std_avg_Nch_3mb)

{'60 < Nch < 71': np.float64(63.535297447228096), '71 < Nch < 78': np.float64(73.33876359279431), '78 < Nch < 91': np.float64(81.67707946063254), '91 < Nch < 97': np.float64(92.97068713073804), '97 < Nch < 1000': np.float64(101.78445284421862)}
{'60 < Nch < 71': np.float64(63.535297447228096), '71 < Nch < 78': np.float64(73.33876359279431), '78 < Nch < 91': np.float64(81.67707946063254), '91 < Nch < 97': np.float64(92.97068713073804), '97 < Nch < 1000': np.float64(101.78445284421862)}
{'60 < Nch < 71': np.float64(63.55845521190132), '71 < Nch < 78': np.float64(73.35382021049458), '78 < Nch < 91': np.float64(81.75718234182547), '91 < Nch < 97': np.float64(93.00658954636303), '97 < Nch < 1000': np.float64(103.80455093132123)}
{'60 < Nch < 71': np.float64(63.55845521190132), '71 < Nch < 78': np.float64(73.35382021049458), '78 < Nch < 91': np.float64(81.75718234182547), '91 < Nch < 97': np.float64(93.00658954636303), '97 < Nch < 1000': np.float64(103.80455093132123)}


In [34]:
wta_projections_0mb = {}
wta_projections_3mb = {}
std_projections_0mb = {}
std_projections_3mb = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    
    wta_projections_0mb[bin_key] = project(wta_yields_0mb[bin_key], f"wta0mb_{mult_bin[0]}_{mult_bin[1]}")
    std_projections_0mb[bin_key] = project(std_yields_0mb[bin_key], f"std0mb_{mult_bin[0]}_{mult_bin[1]}")
    wta_projections_3mb[bin_key] = project(wta_yields_3mb[bin_key], f"wta3mb_{mult_bin[0]}_{mult_bin[1]}")
    std_projections_3mb[bin_key] = project(std_yields_3mb[bin_key], f"std3mb_{mult_bin[0]}_{mult_bin[1]}")

In [35]:
wta_y_vals_0mb = []
wta_y_errs_0mb = []

std_y_vals_0mb = []
std_y_errs_0mb = []

wta_y_vals_3mb = []
wta_y_errs_3mb = []

std_y_vals_3mb = []
std_y_errs_3mb = []

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"

    wta_y_0mb , wta_y_e_0mb = fourierFit(wta_projections_0mb[bin_key], "wta0mb")
    wta_y_vals_0mb.append(wta_y_0mb)
    wta_y_errs_0mb.append(wta_y_e_0mb)

    std_y_0mb , std_y_e_0mb = fourierFit(std_projections_0mb[bin_key], "std0mb")
    std_y_vals_0mb.append(std_y_0mb)
    std_y_errs_0mb.append(std_y_e_0mb)

    wta_y_3mb , wta_y_e_3mb = fourierFit(wta_projections_3mb[bin_key], "wta3mb")
    wta_y_vals_3mb.append(wta_y_3mb)
    wta_y_errs_3mb.append(wta_y_e_3mb)

    std_y_3mb , std_y_e_3mb = fourierFit(std_projections_3mb[bin_key], "std3mb")
    std_y_vals_3mb.append(std_y_3mb)
    std_y_errs_3mb.append(std_y_e_3mb)


In [36]:
print(wta_y_vals_0mb)
print(wta_y_errs_0mb)
print(std_y_vals_0mb)
print(std_y_errs_0mb)
print(wta_y_vals_3mb)
print(wta_y_errs_3mb)
print(std_y_vals_3mb)
print(std_y_errs_3mb)

[0.0012623434757179604, 0.0010626380275079244, 0.000987848739806317, 0.0011529526700064904, 0.0017042700167907608]
[1.2446372018846685e-05, 3.00071819476309e-05, 4.521696930222283e-05, 0.00015711106587734135, 0.00021065596575298986]
[0.017267784597504773, 0.011614871222055491, 0.009231841808259382, 0.006078348837226745, 0.00491927010066692]
[3.6073860926579835e-05, 0.0001047578170378904, 0.0001760601796311062, 0.0006884953984741268, 0.0009532495976952841]
[0.0012235845900829317, 0.0010990018512749946, 0.0008425530312850991, 0.0010602043963129938, 0.00046542050481730046]
[1.7466139377015244e-05, 4.154291134252288e-05, 6.155005091311806e-05, 0.00020469078934488898, 0.00025054039021438733]
[0.018041599258617574, 0.013304485129616623, 0.012362739779048516, 0.012882657861911487, 0.014052264378254629]
[4.8804037490679813e-05, 0.0001349347769964759, 0.0002077480443528131, 0.0006454841583509941, 0.0006494911456500464]


In [37]:
# both 0mb and 3.0mb on the same graph


wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

# CRITICAL: Convert standard Python lists to C-compatible double arrays for PyROOT
wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)

# Initialize TGraphErrors for both configurations
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)

gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration matching the reference
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Winner-Take-All 0mb, open blue square
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColor(ROOT.kBlue)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

# Winner-take-all 3mb, closed blue circle
gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColor(ROOT.kBlue)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

# Standard Axis 0mb, open red square
gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColor(ROOT.kRed)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

# Standard Axis 3mb, closed red circle
gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColor(ROOT.kRed)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)

# Wrap both inside a TMultiGraph to cleanly coordinate unified x and y scaling limits
mg = ROOT.TMultiGraph()
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") # "A" draws the bounding coordinate frame layout

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.05)
mg.GetHistogram().SetMaximum(0.15)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 90, 0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Draw Legend using standard root bounding bounds
legend = ROOT.TLegend(0.48, 0.74, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) # Transparent background
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.Draw()


# Add standardized scientific annotations
#latex = ROOT.TLatex()
#latex.SetNDC()
#latex.SetTextSize(0.033)
#latex.DrawLatex(0.18, 0.42, "#bf{CMS} Pythia 8 Simulation")
#latex.DrawLatex(0.18, 0.37, "pp collisions #sqrt{s} = 13 TeV")
#latex.DrawLatex(0.18, 0.32, "Jet p_{T} > 550 GeV/c, |#eta_{jet}| < 1.6")
#latex.DrawLatex(0.18, 0.27, "Associated 0.3 < j_{T} < 3.0 GeV/c")
#latex.DrawLatex(0.18, 0.22, "2.0 < |#Delta#eta^{*}| < 4.0 (Long-range)")

canvas.Update()
canvas.SaveAs("combined_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_v2_summary_plot.pdf has been created


In [38]:

# same but boxes are in between

# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# NEW: GENERATE THE DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(wta_x_0mb, wta_x_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_mid, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(std_x_0mb, std_x_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_mid, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.04)
mg.GetHistogram().SetMaximum(0.14)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt3_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_alt3_v2_summary_plot.pdf has been created


In [58]:

# same but boxes are in between, axis scaled down

# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_y_vals_0mb)
ey_wta_0mb = array.array('d', wta_y_errs_0mb)
y_std_0mb = array.array('d', std_y_vals_0mb)
ey_std_0mb = array.array('d', std_y_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_y_vals_3mb)
ey_wta_3mb = array.array('d', wta_y_errs_3mb)
y_std_3mb = array.array('d', std_y_vals_3mb)
ey_std_3mb = array.array('d', std_y_errs_3mb)


# =====================================================================
# NEW: GENERATE THE DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(wta_x_0mb, wta_x_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_mid, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(std_x_0mb, std_x_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_mid, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Fourier Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(-0.01)
mg.GetHistogram().SetMaximum(0.04)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_alt4_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_alt4_v2_summary_plot.pdf has been created


In [42]:
def findEllipticFlowCoeff(vals, errs):
    coeffs = []
    coeff_errs = []

    for idx in range(len(vals)):
        coeffs.append(np.sqrt(vals[idx]))
        coeff_errs.append( coeffs[idx] * errs[idx]/vals[idx] )
    
    return coeffs, coeff_errs

In [43]:
wta_v2_0mb, wta_v2_errs_0mb = findEllipticFlowCoeff(wta_y_vals_0mb, wta_y_errs_0mb)

std_v2_0mb, std_v2_errs_0mb = findEllipticFlowCoeff(std_y_vals_0mb, std_y_errs_0mb)

wta_v2_3mb, wta_v2_errs_3mb = findEllipticFlowCoeff(wta_y_vals_3mb, wta_y_errs_3mb)

std_v2_3mb, std_v2_errs_3mb = findEllipticFlowCoeff(std_y_vals_3mb, std_y_errs_3mb)

In [44]:
print(wta_v2_0mb)
print(wta_v2_errs_0mb)
print(std_v2_0mb)
print(std_v2_errs_0mb)
print(wta_v2_3mb)
print(wta_v2_errs_3mb)
print(std_v2_3mb)
print(std_v2_errs_3mb)

[np.float64(0.03552947333859539), np.float64(0.03259812920257732), np.float64(0.03143006108499182), np.float64(0.03395515675131674), np.float64(0.04128280534061077)]
[np.float64(0.0003503111881291606), np.float64(0.0009205185291816815), np.float64(0.0014386535609952858), np.float64(0.004627016362433633), np.float64(0.005102753168418115)]
[np.float64(0.13140694272946454), np.float64(0.10777231194539481), np.float64(0.09608247399114671), np.float64(0.07796376618164842), np.float64(0.07013750851482337)]
[np.float64(0.0002745202055331832), np.float64(0.0009720290411044371), np.float64(0.0018323859942171025), np.float64(0.008830966385974682), np.float64(0.013591152835059966)]
[np.float64(0.03497977401417756), np.float64(0.03315119683020501), np.float64(0.0290267640512183), np.float64(0.03256078003231793), np.float64(0.021573606671516483)]
[np.float64(0.000499321103959565), np.float64(0.001253134586823482), np.float64(0.002120458581070621), np.float64(0.006286421551993684), np.float64(0.0116

In [57]:

# v2 and boxes are in between

# [Existing list extractions and array setups from your code...]
wta_n_points_0mb = len(list(wta_avg_Nch_0mb.values()))
std_n_points_0mb = len(list(std_avg_Nch_0mb.values()))
wta_n_points_3mb = len(list(wta_avg_Nch_3mb.values()))
std_n_points_3mb = len(list(std_avg_Nch_3mb.values()))

wta_x_0mb = array.array('d', list(wta_avg_Nch_0mb.values()))
std_x_0mb = array.array('d', list(std_avg_Nch_0mb.values()))
y_wta_0mb = array.array('d', wta_v2_0mb)
ey_wta_0mb = array.array('d', wta_v2_errs_0mb)
y_std_0mb = array.array('d', std_v2_0mb)
ey_std_0mb = array.array('d', std_v2_errs_0mb)

wta_x_3mb = array.array('d', list(wta_avg_Nch_3mb.values()))
std_x_3mb = array.array('d', list(std_avg_Nch_3mb.values()))
y_wta_3mb = array.array('d', wta_v2_3mb)
ey_wta_3mb = array.array('d', wta_v2_errs_3mb)
y_std_3mb = array.array('d', std_v2_3mb)
ey_std_3mb = array.array('d', std_v2_errs_3mb)


# =====================================================================
# NEW: GENERATE THE DIFFERENCE BOXES
# =====================================================================
# Define the half-width of your grey boxes along the X-axis
box_width = 1.5  

# 1. WTA Difference Bars
wta_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_ex_box = array.array('d', [box_width] * wta_n_points_0mb)
wta_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_wta_0mb, y_wta_3mb)])
wta_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(wta_x_0mb, wta_x_3mb)])

gr_wta_boxes = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_mid, wta_y_mid, wta_ex_box, wta_ey_box)
gr_wta_boxes.SetFillColorAlpha(ROOT.kBlue-10, 0.6)
gr_wta_boxes.SetFillStyle(1001)  # Solid fill style
gr_wta_boxes.SetLineColorAlpha(ROOT.kBlue-10, 0.6)  # Match border to fill color

# 2. Standard Axis Difference Bars
std_y_mid = array.array('d', [(y1 + y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_ex_box = array.array('d', [box_width] * std_n_points_0mb)
std_ey_box = array.array('d', [abs(y1 - y2) / 2.0 for y1, y2 in zip(y_std_0mb, y_std_3mb)])
std_x_mid = array.array('d', [(x1 + x2) / 2.0 for x1, x2 in zip(std_x_0mb, std_x_3mb)])

gr_std_boxes = ROOT.TGraphErrors(std_n_points_0mb, std_x_mid, std_y_mid, std_ex_box, std_ey_box)
gr_std_boxes.SetFillColorAlpha(ROOT.kRed-10, 0.6)
gr_std_boxes.SetFillStyle(1001)
gr_std_boxes.SetLineColorAlpha(ROOT.kRed-10, 0.6)
# =====================================================================


# Initialize TGraphErrors for your markers
gr_wta_0mb = ROOT.TGraphErrors(wta_n_points_0mb, wta_x_0mb, y_wta_0mb, 0, ey_wta_0mb)
gr_std_0mb = ROOT.TGraphErrors(std_n_points_0mb, std_x_0mb, y_std_0mb, 0, ey_std_0mb)
gr_wta_3mb = ROOT.TGraphErrors(wta_n_points_3mb, wta_x_3mb, y_wta_3mb, 0, ey_wta_3mb)
gr_std_3mb = ROOT.TGraphErrors(std_n_points_3mb, std_x_3mb, y_std_3mb, 0, ey_std_3mb)

# Canvas initialization & grid configuration
canvas = ROOT.TCanvas("c_summary", "Elliptic Anisotropy Coefficients vs Multiplicity", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

# Styling Markers (0mb = Open Squares [25], 3mb = Closed Circles [20])
gr_wta_0mb.SetMarkerStyle(25)
gr_wta_0mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_0mb.SetLineColor(ROOT.kBlue)
gr_wta_0mb.SetLineWidth(2)
gr_wta_0mb.SetMarkerSize(1.3)

gr_wta_3mb.SetMarkerStyle(20)
gr_wta_3mb.SetMarkerColorAlpha(ROOT.kBlue, 0.6)
gr_wta_3mb.SetLineColor(ROOT.kBlue)
gr_wta_3mb.SetLineWidth(2)
gr_wta_3mb.SetMarkerSize(1.3)

gr_std_0mb.SetMarkerStyle(25) 
gr_std_0mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_0mb.SetLineColor(ROOT.kRed)
gr_std_0mb.SetLineWidth(2)
gr_std_0mb.SetMarkerSize(1.3)

gr_std_3mb.SetMarkerStyle(20) 
gr_std_3mb.SetMarkerColorAlpha(ROOT.kRed, 0.6)
gr_std_3mb.SetLineColor(ROOT.kRed)
gr_std_3mb.SetLineWidth(2)
gr_std_3mb.SetMarkerSize(1.3)


# =====================================================================
# CRITICAL LAYER MANAGEMENT IN TMULTIGRAPH
# =====================================================================
mg = ROOT.TMultiGraph()

# Add the grey bars FIRST so they sit in the background layer
mg.Add(gr_wta_boxes, "2")
mg.Add(gr_std_boxes, "2")

# Add the data markers NEXT so they are drawn cleanly on top of the bars
mg.Add(gr_wta_0mb, "P")
mg.Add(gr_std_0mb, "P")
mg.Add(gr_wta_3mb, "P")
mg.Add(gr_std_3mb, "P")
# =====================================================================


mg.SetTitle("High multiplicity datasets, with and without hFSI; Jet Multiplicity N_{ch}^{j}; Elliptic Anisotropy Coefficient #font[12]{v_{2}^{*}}")
mg.Draw("A") 

# Enforce strict axis scaling configurations
mg.GetXaxis().SetLimits(0, 120)
mg.GetHistogram().SetMinimum(0)
mg.GetHistogram().SetMaximum(0.25)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Render a baseline guide at v2 = 0
line = ROOT.TLine(0, 0, 120, 0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Legend setup
legend = ROOT.TLegend(0.48, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(gr_wta_3mb, "WTA Axis (3.0mb)", "pe")
legend.AddEntry(gr_wta_0mb, "WTA Axis (0mb)", "pe")
legend.AddEntry(gr_std_3mb, "Standard Axis (3.0mb)", "pe")
legend.AddEntry(gr_std_0mb, "Standard Axis (0mb)", "pe")
legend.AddEntry(gr_wta_boxes, "WTA difference", "f")
legend.AddEntry(gr_std_boxes, "Standard difference", "f")
legend.Draw()

canvas.Update()
canvas.SaveAs("combined_coeff_v2_summary_plot.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file combined_coeff_v2_summary_plot.pdf has been created
